# 01. The iteration protocol

**Tier:** Deep. **Why you'd use this:** A `for` loop is sugar over `iter` and `next`. Generators and `yield from` matter when you hand a stretch of values to another iterator without building a list.

**After this lesson you can:** rewrite a `for` loop as get-iterator then get-next-value; explain why a generator is empty the second time you loop it; pass a stretch of values to another generator with `yield from`.
About 30 minutes. **Uses:** [Loops](../../course/03-control-flow/02-loops-lesson.ipynb), [Generators](../../course/07-iteration/02-generators-lesson.ipynb), [Scopes and closures](../06-scopes/01-closures-lesson.ipynb)


## Quick review

1. What does a closure remember?
2. Why can a loop variable surprise you inside functions created in that loop?

<details><summary>Answers</summary>

Enclosing names it reads later. All those functions often share the same name binding, which holds the last value.
</details>


## The idea

`for` does not know what a list is. It asks the object for an iterator, then asks that iterator for the next value until the iterator raises `StopIteration`. The container and the cursor are different objects. Two loops over one list each get their own cursor. A generator is a cursor that produces values, and it is also its own iterator, so it can be walked only once.

`yield` suspends the generator's frame. `yield from` hands the rest of the walk to another iterator and does not resume until that iterator is exhausted.

<details><summary>Explain more</summary>

`iter(obj)` calls `obj.__iter__`. `next(iterator)` calls `iterator.__next__`. A list's `__iter__` returns a new iterator positioned at the start. A generator's `__iter__` returns the generator itself.

`generator.gi_frame` is the suspended frame, the same kind of frame a call would use. It becomes `None` when the generator is exhausted.
</details>

<details><summary>If you already write Java, JavaScript, or C#</summary>

This is `Iterator` / `IEnumerator`. `StopIteration` is the end signal, not a value. A generator is a coroutine that only produces, and `yield from` is like delegating to another enumerator.
</details>


## See it work

A log filter should return the first positive reading and ignore the rest, without building a second list. Written by hand, that is `iter` plus `next` until `StopIteration`. The `for` statement is this loop with the exception hidden.


In [1]:
def first_positive(readings: list[int]) -> int | None:
    # 1. A cursor at the start. The list itself is not the cursor.
    cursor = iter(readings)
    while True:
        # 2. StopIteration is the end, not a missing reading.
        try:
            value = next(cursor)
        except StopIteration:
            return None
        # 3. The first positive value is the answer.
        if value > 0:
            return value


print(first_positive([-1, 0, 4, 9]))
print(first_positive([-2, 0]))
print(first_positive([]))


4
None
None


| step | cursor position | action |
| --- | --- | --- |
| start | before -1 | `iter` |
| next | -1 | not positive |
| next | 0 | not positive |
| next | 4 | return 4 |

```
readings: [-1, 0, 4, 9]
cursor:    ^
for is: iter, then next until StopIteration
```

The values after 4 are never pulled. A list comprehension would have walked all of them.


## Check yourself

Answer these from the example above before you run the cell.


1. Why is the second `list(seq)` empty even though the function's source still says `yield`?
2. Why can two `for` loops walk one list, when they cannot walk one generator?
3. What does `gi_frame is None` tell you that `list(seq)` also tells you, more cheaply to inspect?

<details><summary>Answers</summary>

1. The source describes how to build a new generator. `seq` is one generator whose frame has already returned. Asking it for more values hits the end immediately.
2. Each `for` on a list calls `iter`, and the list returns a fresh iterator with its own index. A generator's `iter` returns the same object.
3. The frame has been thrown away. The generator is exhausted. `list(seq)` would show that by being empty, and it would also pull any values that were still waiting. `gi_frame` does not pull them.
</details>


In [2]:
def countdown(n: int):
    while n:
        yield n
        n -= 1


seq = countdown(2)
print(list(seq))
print(list(seq))


[2, 1]
[]


<details><summary>Check your answers</summary>

- The first `list` is `[2, 1]`. The generator ran until it fell off the end.
- The second `list` is `[]`. The same generator is already exhausted. `list` asks for a new iterator, and a generator returns itself, still finished. Nothing in the second call restarts `n`.
</details>


## Why it works this way

The protocol is two methods so the container does not have to remember a cursor. A list can be in two loops at once because each loop called `__iter__` and got a separate iterator, each holding an index. If the list itself were the iterator, the second loop would continue where the first stopped, or the two loops would fight over one index. Separating them costs one small iterator object per loop. That object is the position. The list is the data.

`for` is sugar for the `while` in `first_positive`. It calls `iter` once, then `next` in a loop, and it treats `StopIteration` as "break", not as a failure to show the user. Any other exception propagates. That is why a bug inside the loop still traceback normally, and why the end of the data does not. Raising `StopIteration` yourself inside a generator is the wrong signal: a generator turns that into `RuntimeError`, so the way to finish a generator is to `return`, which the machinery converts into `StopIteration` for the caller of `next`.

A generator function does not run at the call. `countdown(2)` returns a generator object whose frame is suspended before the first line. The first `next` runs until `yield`, sends that value out, and leaves the frame parked, with local `n` still in it. `gi_frame` is that frame. The next `next` resumes after the `yield`. When the function returns, the frame is discarded and `gi_frame` becomes `None`. The generator cannot be rewound because the frame is gone. Memory while it is alive is one frame plus the objects it currently points at, not a list of every value it will ever yield. Time to pull n values is still proportional to n. The saving is the list you did not allocate.

`yield from other` is a nested use of the same protocol. The outer generator's `next` calls `next` on `other` until `other` raises `StopIteration`, then continues after the `yield from`. You do not write that loop. Values from `other` are yielded as the outer generator's own values. A recursive flatten uses this to walk nested lists without building the flat list first, and the frames are the generator frames, one per nesting level, the same depth concern as the recursion unit.

The design reason is a single shape for lists, files, generators, and your own cursors. `for` stays small. The object decides how a cursor works. The cost of that uniformity is the exhausted-generator surprise: the second loop is not a second cursor, because the generator refused to make one.

Before the first `next`, the frame exists and is suspended. After the last `yield` it is still there, parked after that yield. One more `next` runs off the end, the function returns, and `gi_frame` becomes `None`. A second call builds a new frame and walks again.


In [3]:
def steps(n: int):
    while n:
        yield n
        n -= 1


seq = steps(2)
print("started", seq.gi_frame is not None)
print(next(seq), next(seq))
print("after yields", seq.gi_frame is not None)
try:
    next(seq)
except StopIteration:
    print("finished", seq.gi_frame is None)
print(list(steps(2)))


started True
2 1
after yields True
finished True
[2, 1]


In [4]:
def chain(parts: list[list[int]]):
    for part in parts:
        yield from part


print(list(chain([[1, 2], [3]])))


[1, 2, 3]


## When to use it

Use a generator when the consumer might stop early, or when the sequence would be large to store. Use a list when you will walk the values more than once, index them, or hand them to something that measures `len`. Use `yield from` when one generator is a concatenation of others, instead of a loop that `yield`s each item by hand.

Do not pass a generator into two consumers. Materialize with `list` once if both need it, or call the generator function twice if it is cheap and pure. Do not catch `StopIteration` in ordinary code. `for` already does. Catch it only when you are writing the protocol yourself, as `first_positive` does.

A file object is an iterator of lines. The same one-pass rule applies. Read it again by reopening, not by looping the same object twice.


## Common mistake

Saving one generator and looping it twice looks like saving a list. The second loop prints nothing. The function is fine. The cursor was already finished. Nothing raises, so a test that only checks the first consumer stays green.


In [5]:
def positives(readings: list[int]):
    for reading in readings:
        if reading > 0:
            yield reading


seq = positives([-1, 4, 0, 9])
print(list(seq))
print(list(seq))


[4, 9]
[]


The prints are `[4, 9]` and `[]`. The fix calls `positives` again for the second consumer. Each call builds a new generator and a new frame. The list of readings is unchanged and can be walked any number of times, because each `for` inside `positives` calls `iter` on it again.


In [6]:
def positives_again(readings: list[int]):
    for reading in readings:
        if reading > 0:
            yield reading


readings = [-1, 4, 0, 9]
print(list(positives_again(readings)))
print(list(positives_again(readings)))


[4, 9]
[4, 9]


## Try it

Change `first_positive` so it stops by raising `StopIteration` from inside a generator instead of returning `None` from a plain function. What exception does the caller of `next` actually see, and why is `return` the way a generator is supposed to end?


## What this explains

- [Loops](../../course/03-control-flow/02-loops-lesson.ipynb): `for` works on anything with the protocol. `range` is lazy because it returns an iterator, not a list of every integer.
- [Generators](../../course/07-iteration/02-generators-lesson.ipynb): `yield` suspends a frame. The one-pass behavior is `gi_frame` ending at `None`.
- [Scopes and closures](../06-scopes/01-closures-lesson.ipynb): the generator's locals live in `gi_frame`, a real frame, which is why they survive from one `next` to the next and disappear when the generator returns.


## Recap

- `for` is `iter` once, then `next` until `StopIteration`.
- A list grows a new cursor per loop. A generator is its cursor, and the frame is gone when it ends.
- `yield from` runs that same protocol on another iterator.

Next: [Attribute lookup](../08-attributes/01-lookup-lesson.ipynb).
